# Use cases
## 1. Loading a model

`read_parts` loads the full B-Rep structure into memory as linked Python objects.
Each part contains its geometry, topology, and optional mesh.

In [14]:
from abs.utils import read_parts

FILE_PATH = '../data/sample_hdf5/Cylinder.hdf5'
parts = read_parts(FILE_PATH)

print(f"Number of parts: {len(parts)}")

part = parts[0]
print(f"Number of faces:     {len(part.faces)}")
print(f"Number of edges:     {len(part.edges)}")
print(f"Number of halfedges: {len(part.halfedges)}")
print(f"Number of loops:     {len(part.loops)}")
print(f"Number of shells:    {len(part.shells)}")
print(f"Number of solids:    {len(part.solids)}")

Number of parts: 1
Number of faces:     3
Number of edges:     3
Number of halfedges: 6
Number of loops:     3
Number of shells:    1
Number of solids:    1


---
## 2. Traversing topology

You can navigate the B-Rep structure directly through Python objects.

In [15]:
face = part.faces[0]

# Surface type attached to this face
print(f"Surface type: {face.surface.shape_name}")
print(f"Surface orientation: {face.surface_orientation}")
print(f"Number of loops: {len(face.loops)}")

# Iterate over the halfedges in the first loop
first_loop = face.loops[0]
print(f"\nHalfedges in first loop: {len(first_loop.halfedges)}")
for he in first_loop.halfedges:
    print(f"  Halfedge {he.id} — edge {he.edge.id}, curve: {he.edge.curve3d.shape_name}")

Surface type: Cylinder
Surface orientation: True
Number of loops: 1

Halfedges in first loop: 4
  Halfedge 0 — edge 0, curve: Circle
  Halfedge 1 — edge 1, curve: BSpline
  Halfedge 2 — edge 2, curve: Circle
  Halfedge 3 — edge 1, curve: BSpline


---
## 3. Inspecting geometry

Each face references a parametric surface. You can access its type,
parametric domain, and geometric attributes directly.

In [16]:
for i, face in enumerate(part.faces):
    surface = face.surface
    print(f"Face {i:2d} — surface type: {surface.shape_name:12s}, "
          f"UV domain: u=[{surface.trim_domain[0,0]:.2f}, {surface.trim_domain[0,1]:.2f}] "
          f"v=[{surface.trim_domain[1,0]:.2f}, {surface.trim_domain[1,1]:.2f}]")

Face  0 — surface type: Cylinder    , UV domain: u=[3.14, 9.42] v=[0.00, 25.40]
Face  1 — surface type: Plane       , UV domain: u=[-44.94, 44.94] v=[-44.94, 44.94]
Face  2 — surface type: Plane       , UV domain: u=[-44.94, 44.94] v=[-44.94, 44.94]


---
## 4. Computing normals

Normals are evaluated directly from the parametric surface.
The callback receives the face and the sampled UV points, and returns per-point normals.

In [30]:
from abs.part_processor import sample_parts

def get_normals(face, points):
    return face.normal(points)

face_points, face_normals, _, _ = sample_parts(
    parts, num_samples=5000, face_func=get_normals
)

print(f"Points shape for first part:  {face_points[0].shape}")
print(f"Normals shape for first part: {face_normals[0].shape}")
print(f"\nFirst 3 points:\n{face_points[0][:3]}")
print(f"\nFirst 3 normals:\n{face_normals[0][:3]}")

Points shape for first part:  (5000, 3)
Normals shape for first part: (5000, 3)

First 3 points:
[[ 32.63568526 -30.89042724   8.82354873]
 [-44.43727533  -6.68094365  15.12717921]
 [-11.89046015  43.33501361   7.480638  ]]

First 3 normals:
[[ 0.72625916 -0.68742101  0.        ]
 [-0.98888618 -0.14867457 -0.        ]
 [-0.2646047   0.96435696  0.        ]]


---
## 5. Sampling points with user-defined labels

`sample_parts` generates points on parametric surfaces and curves and passes them
to your callbacks. Each callback returns whatever per-point values your task needs.
Return `None` to skip an entity.

Here we assign label `0` to face points and label `1` to edge points.

In [32]:
import numpy as np

def face_func(face, points):
    return np.zeros(points.shape[0])   # label 0 for faces

def edge_func(edge, points):
    return np.ones(points.shape[0])    # label 1 for edges

face_points, face_labels, edge_points, edge_labels = sample_parts(
    parts, num_samples=5000, face_func=face_func, edge_func=edge_func
)

print(f"Face points: {face_points[0].shape}, labels: {np.unique(face_labels[0])}")
print(f"Edge points: {edge_points[0].shape}, labels: {np.unique(edge_labels[0])}")


Face points: (4711, 3), labels: [0.]
Edge points: (289, 3), labels: [1.]


---
## 6. Primitive type segmentation

Surface types (plane, cylinder, cone, etc.) are directly accessible from the B-Rep
without any heuristic reconstruction. Here we assign a unique integer label per surface type.

In [33]:
SURFACE_TYPES = {
    'Plane': 0, 'Cylinder': 1, 'Cone': 2, 'Sphere': 3,
    'Torus': 4, 'BSpline': 5, 'Extrusion': 6, 'Revolution': 7,
    'Offset': 8, 'Other': 9
}

def primitive_label(face, points):
    label = SURFACE_TYPES.get(face.surface.shape_name, 9)
    return np.full(points.shape[0], label)

face_points, face_labels, _, _ = sample_parts(
    parts, num_samples=5000, face_func=primitive_label
)

unique, counts = np.unique(face_labels, return_counts=True)
inv_map = {v: k for k, v in SURFACE_TYPES.items()}
print("Surface type distribution:")
for label, count in zip(unique, counts):
    print(f"  {inv_map[int(label)]}: {count} points")

Surface type distribution:
  Plane: 3188 points
  Cylinder: 1812 points


---
## 7. Extracting the mesh

The mesh is stored independently from the B-Rep topology.
`get_mesh` concatenates all per-face triangulations into a single consistent mesh.

In [34]:
from abs.utils import read_meshes, get_mesh

meshes = read_meshes(FILE_PATH)
V, F = get_mesh(meshes)

print(f"Vertices: {V.shape}")
print(f"Faces:    {F.shape}")


Vertices: (286, 3)
Faces:    (280, 3)
